# Rede neural (MLP) contra XGBoost: a AUC aumenta?

**Pergunta:** trocar o XGBoost por uma rede neural densa melhora a AUC de "estadia longa" nas primeiras 24 horas?

**O que este caderno faz**

- Usa as variantes **B** (todas as estadias de UTI com IC, com as variáveis novas) e **D** (primeira UTI e sem óbitos hospitalares, com as variáveis novas). São as mesmas do caderno `variaveis_24h_e_coorte_yang.ipynb`.
- Lê a tabela em cache `artefatos_24h_ampliado/tabela_24h_ampliada.csv`. Rode antes aquele caderno, se o arquivo não existir.
- Treina `MLPClassifier` (scikit-learn) e `XGBClassifier` com a mesma divisão por paciente (`random_state=42`, 80/20), no mesmo teste.
- Compara a AUC com intervalo de confiança de 95% por reamostragem de pacientes.

**Cuidados com a rede neural**

- Variáveis muito assimétricas (exames, PEEP) passam por `log(1 + x)`. Depois, tudo é padronizado com média e desvio do treino.
- A arquitetura e a regularização (`alpha`) são escolhidas só no treino, com `GroupKFold` por paciente e AUC.
- A previsão final é a média de 5 redes com sementes diferentes, para reduzir a variação do sorteio inicial.
- O teste só mede o resultado final.

**Regras que continuam valendo:** o CID só define a coorte; nada depois de `intime + 24 h` entra em X; imputação, one-hot e escala são ajustados só no treino.

## 1. Configuração e dados

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, GroupKFold, GroupShuffleSplit
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

RANDOM_STATE = 42
RAIZ = Path.cwd()
CACHE = RAIZ / "artefatos_24h_ampliado" / "tabela_24h_ampliada.csv"
SAIDA = RAIZ / "artefatos_rede_neural"
SAIDA.mkdir(exist_ok=True)
CORTES_DIAS = [3.55, 7.0]
N_BOOTSTRAP = 200
N_REDES = 5

BASE_NUM = [
    "idade", "pam_24h", "frequencia_cardiaca_24h", "frequencia_respiratoria_24h", "spo2_24h",
    "creatinina_24h", "ureia_24h", "sodio_24h", "potassio_24h", "glicose_24h", "ptt_24h", "feve",
]
BASE_BIN = ["feve_ausente"]
NOVAS_NUM = [
    "temperatura_24h", "gcs_min_24h", "fio2_max_24h", "peep_max_24h",
    "lactato_max_24h", "bicarbonato_min_24h", "leucocitos_max_24h", "hemoglobina_min_24h",
    "plaquetas_min_24h", "inr_max_24h", "bilirrubina_max_24h", "nt_probnp_max_24h",
    "troponina_t_max_24h", "ph_min_24h", "pco2_max_24h",
]
NOVAS_BIN = [
    "ventilacao_invasiva_24h", "lactato_medido", "nt_probnp_medido", "troponina_medida", "gasometria_medida",
]
CATEGORICAS = ["genero", "tipo_admissao"]
# Variáveis positivas e assimétricas: log(1 + x) ajuda a rede neural.
ASSIMETRICAS = [
    "creatinina_24h", "ureia_24h", "glicose_24h", "ptt_24h", "peep_max_24h", "lactato_max_24h",
    "leucocitos_max_24h", "plaquetas_min_24h", "inr_max_24h", "bilirrubina_max_24h",
    "nt_probnp_max_24h", "troponina_t_max_24h",
]

assert CACHE.exists(), "Rode antes o caderno variaveis_24h_e_coorte_yang.ipynb para gerar o cache."
tabela = pd.read_csv(CACHE, parse_dates=["intime"])
tabela["lactato_medido"] = tabela["lactato_max_24h"].notna().astype(int)
tabela["nt_probnp_medido"] = tabela["nt_probnp_max_24h"].notna().astype(int)
tabela["troponina_medida"] = tabela["troponina_t_max_24h"].notna().astype(int)
tabela["gasometria_medida"] = tabela["ph_min_24h"].notna().astype(int)
tabela["ventilacao_invasiva_24h"] = tabela["ventilacao_invasiva_24h"].fillna(0).astype(int)
tabela = tabela.dropna(subset=["los_dias", "subject_id"])
tabela = tabela[tabela["los_dias"] > 0].copy()

todas = tabela.copy()
primeira = tabela.sort_values("intime").drop_duplicates("subject_id", keep="first")
yang = primeira[primeira["hospital_expire_flag"] == 0].copy()
print(f"Todas: {len(todas):,} estadias | Yang: {len(yang):,} estadias")

## 2. Preparação, intervalo de confiança e modelos

O XGBoost usa os mesmos hiperparâmetros do caderno anterior. A rede neural recebe os dados com `log(1 + x)` nas variáveis assimétricas e padronização.

In [ ]:
def preparar(base, colunas_num, colunas_bin):
    divisor = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
    i_treino, i_teste = next(divisor.split(base, groups=base["subject_id"]))
    treino, teste = base.iloc[i_treino].copy(), base.iloc[i_teste].copy()
    assert not set(treino["subject_id"]) & set(teste["subject_id"]), "paciente repetido"

    for c in CATEGORICAS:
        treino[c] = treino[c].astype("string").fillna("desconhecido")
        teste[c] = teste[c].astype("string").fillna("desconhecido")
    try:
        codificador = OneHotEncoder(sparse_output=False, handle_unknown="ignore", dtype=np.float64)
    except TypeError:
        codificador = OneHotEncoder(sparse=False, handle_unknown="ignore", dtype=np.float64)
    d_treino = codificador.fit_transform(treino[CATEGORICAS])
    nomes = codificador.get_feature_names_out(CATEGORICAS)
    d_treino = pd.DataFrame(d_treino, columns=nomes, index=treino.index)
    d_teste = pd.DataFrame(codificador.transform(teste[CATEGORICAS]), columns=nomes, index=teste.index)

    imputador = SimpleImputer(strategy="median")
    n_treino = pd.DataFrame(imputador.fit_transform(treino[colunas_num]), columns=colunas_num, index=treino.index)
    n_teste = pd.DataFrame(imputador.transform(teste[colunas_num]), columns=colunas_num, index=teste.index)

    x_treino = pd.concat([n_treino, treino[colunas_bin].astype(float), d_treino], axis=1)
    x_teste = pd.concat([n_teste, teste[colunas_bin].astype(float), d_teste], axis=1)
    x_teste = x_teste.reindex(columns=x_treino.columns, fill_value=0.0)
    proibidas = [c for c in x_treino.columns if "los" in c or "icd" in c.lower() or "diag" in c.lower()]
    assert not proibidas, f"coluna proibida em X: {proibidas}"
    return treino, teste, x_treino, x_teste


def para_rede(x_treino, x_teste, colunas_num):
    treino, teste = x_treino.copy(), x_teste.copy()
    for c in [c for c in ASSIMETRICAS if c in treino.columns]:
        treino[c] = np.log1p(treino[c].clip(lower=0))
        teste[c] = np.log1p(teste[c].clip(lower=0))
    escala = StandardScaler()
    treino[colunas_num] = escala.fit_transform(treino[colunas_num])
    teste[colunas_num] = escala.transform(teste[colunas_num])
    return treino, teste


def intervalo_auc(y, prob, grupos, repeticoes=N_BOOTSTRAP):
    grupos = np.asarray(grupos)
    ordem = np.argsort(grupos, kind="stable")
    _, inicio = np.unique(grupos[ordem], return_index=True)
    por_paciente = np.split(ordem, inicio[1:])
    sorteio = np.random.default_rng(RANDOM_STATE)
    aucs = []
    for _ in range(repeticoes):
        escolha = sorteio.integers(0, len(por_paciente), len(por_paciente))
        indices = np.concatenate([por_paciente[i] for i in escolha])
        if y[indices].min() == y[indices].max():
            continue
        aucs.append(roc_auc_score(y[indices], prob[indices]))
    return tuple(np.percentile(aucs, [2.5, 97.5]))


def nova_rede(camadas, alpha, semente):
    return MLPClassifier(
        hidden_layer_sizes=camadas, alpha=alpha, activation="relu", solver="adam",
        learning_rate_init=1e-3, batch_size=256, max_iter=200,
        early_stopping=True, validation_fraction=0.15, n_iter_no_change=10,
        random_state=semente,
    )


def avaliar_variante(rotulo, base, colunas_num, colunas_bin):
    treino, teste, x_treino, x_teste = preparar(base, colunas_num, colunas_bin)
    r_treino, r_teste = para_rede(x_treino, x_teste, colunas_num)
    grupos = treino["subject_id"].to_numpy()

    # Arquitetura e regularização escolhidas só no treino, no corte de 3,55 dias.
    y_busca = (treino["los_dias"] > CORTES_DIAS[0]).astype(int).to_numpy()
    busca = GridSearchCV(
        nova_rede((64, 32), 1e-2, RANDOM_STATE),
        {"hidden_layer_sizes": [(64, 32), (128, 64)], "alpha": [1e-2, 1e-1]},
        cv=GroupKFold(n_splits=5), scoring="roc_auc", n_jobs=1,
    )
    busca.fit(r_treino, y_busca, groups=grupos)
    camadas, alpha = busca.best_params_["hidden_layer_sizes"], busca.best_params_["alpha"]
    print(f"{rotulo}: melhor rede {camadas}, alpha {alpha}", flush=True)

    linhas, probs_teste = [], {}
    for corte in CORTES_DIAS:
        y_treino = (treino["los_dias"] > corte).astype(int).to_numpy()
        y_teste = (teste["los_dias"] > corte).astype(int).to_numpy()

        xgb = XGBClassifier(
            n_estimators=400, max_depth=4, learning_rate=0.03, subsample=0.8, colsample_bytree=0.8,
            min_child_weight=5, scale_pos_weight=float((y_treino == 0).sum() / (y_treino == 1).sum()),
            objective="binary:logistic", eval_metric="auc", n_jobs=1, random_state=RANDOM_STATE,
        )
        xgb.fit(x_treino, y_treino)
        prob_xgb = xgb.predict_proba(x_teste)[:, 1]

        redes = [nova_rede(camadas, alpha, RANDOM_STATE + s).fit(r_treino, y_treino) for s in range(N_REDES)]
        prob_rede = np.mean([r.predict_proba(r_teste)[:, 1] for r in redes], axis=0)
        prob_media = (prob_xgb + prob_rede) / 2

        for modelo, prob in [("XGBoost", prob_xgb), ("Rede neural (MLP)", prob_rede), ("Média dos dois", prob_media)]:
            baixo, alto = intervalo_auc(y_teste, prob, teste["subject_id"])
            linhas.append({
                "variante": rotulo, "tarefa": f"mais de {corte} dias", "modelo": modelo,
                "AUC": roc_auc_score(y_teste, prob), "AUC_IC95_baixo": baixo, "AUC_IC95_alto": alto,
                "AUC_PR": average_precision_score(y_teste, prob), "n_teste": len(teste),
            })
            probs_teste[(corte, modelo)] = prob
    return linhas, camadas, alpha

## 3. Treino e teste nas variantes B e D

In [ ]:
linhas_total, arquiteturas = [], {}
for rotulo, base in [("B: todas, com novas variáveis", todas), ("D: Yang, com novas variáveis", yang)]:
    print(f"Treinando {rotulo}...", flush=True)
    linhas, camadas, alpha = avaliar_variante(rotulo, base, BASE_NUM + NOVAS_NUM, BASE_BIN + NOVAS_BIN)
    linhas_total.extend(linhas)
    arquiteturas[rotulo] = {"camadas": list(camadas), "alpha": alpha}

resultados = pd.DataFrame(linhas_total)
resultados.to_csv(SAIDA / "resultados_rede_neural.csv", index=False, encoding="utf-8-sig")

## 4. Resultados

In [ ]:
tabela_auc = resultados.copy()
tabela_auc["AUC (IC 95%)"] = tabela_auc.apply(
    lambda l: f"{l['AUC']:.3f} ({l['AUC_IC95_baixo']:.3f} a {l['AUC_IC95_alto']:.3f})", axis=1
)
display(tabela_auc[["variante", "tarefa", "modelo", "n_teste", "AUC (IC 95%)", "AUC_PR"]].round(3).reset_index(drop=True))
print("Arquiteturas escolhidas no treino:", arquiteturas)

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for eixo, corte in zip(eixos, CORTES_DIAS):
    parte = resultados[resultados["tarefa"] == f"mais de {corte} dias"].reset_index(drop=True)
    rotulos = [f"{v[0]} | {m}" for v, m in zip(parte["variante"], parte["modelo"])]
    y = np.arange(len(parte))[::-1]
    erro = [parte["AUC"] - parte["AUC_IC95_baixo"], parte["AUC_IC95_alto"] - parte["AUC"]]
    eixo.errorbar(parte["AUC"], y, xerr=erro, fmt="o", color="#1f4e79", capsize=4)
    eixo.set_yticks(y)
    eixo.set_yticklabels(rotulos)
    eixo.axvline(0.5, color="gray", linestyle="--", linewidth=1)
    eixo.set_title(f"Estadia > {corte} dias")
    eixo.set_xlabel("AUC no teste, com IC 95%")
plt.tight_layout()
plt.savefig(SAIDA / "auc_rede_neural_contra_xgboost.png", dpi=150)
plt.show()

## 5. Como ler

- Se os intervalos de confiança do **XGBoost** e da **rede neural** se sobrepõem bastante, a diferença pode ser só acaso. Nesse caso a conclusão é que a rede neural **não melhorou** o resultado.
- A **média dos dois** mostra se combinar os modelos ajuda. Em geral o ganho é pequeno.
- Com cerca de 26 mil estadias e poucas dezenas de variáveis, o esperado é que a rede neural fique perto do XGBoost, não muito acima.
- A letra **B** é a coorte inteira. A letra **D** é a de Yang et al. (primeira UTI, sem óbitos hospitalares).
- O que mais aumentou a AUC foi trocar as variáveis (de 0,68 para 0,76 e 0,79), não o modelo.